In [ ]:
# Import Dataset
from google.colab import files
uploaded = files.upload()
import sqlite3
import pandas as pd
df = pd.read_csv('dairy_dataset.csv')

# Dataset Memory
conn = sqlite3.connect(':memory:')
df.to_sql('billing', conn, index=False, if_exists='replace')
print(f"Data berhasil diimport: {len(df)} baris")

Saving dairy_dataset.csv to dairy_dataset (1).csv
Data berhasil diimport: 1000 baris


In [ ]:
# query1: Cek baris dengan data kosong/hilang
query1 = '''
SELECT Customer_ID, Customer_Name, Payment_Mode, Payment_Received
FROM billing
WHERE Payment_Mode IS NULL OR Payment_Received IS NULL
'''
pd.read_sql(query1, conn)

,Customer_ID,Customer_Name,Payment_Mode,Payment_Received
0,C1008,Yash Solanki,None,3840.38
1,C1012,Bharat Kansara,UPI,NaN
2,C1013,Dilip Parmar,None,1928.99
3,C1015,Yash Gohil,UPI,NaN
4,C1018,Hitesh Rabari,Bank Transfer,NaN
5,C1024,Piyush Trivedi,None,3548.01
6,C1028,Hitesh Mehta,None,860.73
7,C1029,Kalpesh Kansara,UPI,NaN
8,C1033,Nilesh Rabari,None,793.19
9,C1033,Nilesh Rabari,Bank Transfer,NaN


In [ ]:
# query2: Cek baris dengan data duplikat persis sama
query2 = '''
SELECT Customer_ID, Customer_Name, Month, Year, COUNT(*) AS jumlah_duplikat
FROM billing
GROUP BY Customer_ID, Customer_Name, Month, Year, Bill_Amount
HAVING COUNT(*) > 1
'''
pd.read_sql(query2, conn)

,Customer_ID,Customer_Name,Month,Year,jumlah_duplikat
0,C1002,Ankit Parmar,May,2026,2
1,C1025,Ajay Bhatti,February,2026,2
2,C1029,Kalpesh Kansara,March,2026,2
3,C1032,Krunal Makwana,April,2026,2
4,C1035,Alpesh Chauhan,June,2026,2
5,C1042,Kalpesh Vyas,May,2026,2
6,C1059,Dilip Gohil,February,2026,2
7,C1092,Piyush Patel,February,2026,2
8,C1095,Alpesh Thakkar,April,2026,2
9,C1099,Bhavesh Mehta,April,2026,2


In [6]:
# query3: Merapikan penulisan data agar seragam
query3 = '''
SELECT DISTINCT
    Payment_Mode AS data_awal,
    UPPER(TRIM(Payment_Mode)) AS data_hasil
FROM billing
WHERE Payment_Mode IS NOT NULL
'''
pd.read_sql(query3, conn)

,data_awal,data_hasil
0,Bank Transfer,BANK TRANSFER
1,UPI,UPI
2,Cash,CASH
3,upi,UPI
4,Upi,UPI


In [7]:
# query4: Ringkasan pembayaran per metode SETELAH dirapikan
query4 = '''
SELECT
    UPPER(TRIM(Payment_Mode)) AS metode_pembayaran,
    COUNT(*) AS jumlah_transaksi,
    ROUND(SUM(Payment_Received), 2) AS total_diterima
FROM billing
WHERE Payment_Mode IS NOT NULL AND Payment_Received IS NOT NULL
GROUP BY UPPER(TRIM(Payment_Mode))
ORDER BY total_diterima DESC
'''
pd.read_sql(query4, conn)

,metode_pembayaran,jumlah_transaksi,total_diterima
0,UPI,362,1048266.37
1,BANK TRANSFER,295,875975.48
2,CASH,297,845228.32
